# DEE — Vacío cuántico y constante cosmológica

**Contexto:** en el notebook de termodinámica (DEE_termo_defectos) calculamos la energía de punto cero del cristal como E_0 por modo ≈ 18.8 en unidades naturales. Ahora traducimos eso a unidades físicas y comparamos con la constante cosmológica observada Λ_obs.

**Pregunta central:** si identificamos la energía de punto cero del cristal DEE con la densidad de energía de vacío cosmológico, ¿qué valor obtenemos y cómo se compara con Λ_obs?

**Dos caminos:**

1. **Análisis paramétrico:** ρ_Λ como función de ℓ_c (espaciado de red) y c_s (velocidad del sonido)
2. **Evaluación directa:** asumir ℓ_c = ℓ_Planck y c_s = c, calcular y comparar

**Advertencia epistemológica:** el problema de la constante cosmológica tiene ~120 órdenes de magnitud de discrepancia en QFT estándar y no está resuelto desde hace décadas. Este notebook NO pretende resolverlo. Solo verificamos qué predice DEE cuantitativamente y en qué orden de magnitud cae.

---

In [ ]:
import os
import numpy as np
from scipy.sparse import csr_matrix, diags
from scipy.sparse.linalg import eigsh
import matplotlib.pyplot as plt
import time

PLOT_DIR = 'plots_vacio_Lambda'
os.makedirs(PLOT_DIR, exist_ok=True)

def save_plot(name, fig=None, dpi=120):
    if fig is None: fig = plt.gcf()
    path = os.path.join(PLOT_DIR, f'{name}.png')
    fig.savefig(path, dpi=dpi, bbox_inches='tight')
    print(f'  → {path}')

print('Setup listo')

## Constantes fundamentales y valor observado de Λ

In [ ]:
# Constantes fundamentales (SI)
c_SI = 2.998e8          # velocidad de la luz [m/s]
hbar_SI = 1.055e-34     # ℏ [J·s]
G_SI = 6.674e-11        # Newton [m³/(kg·s²)]
k_B_SI = 1.381e-23      # Boltzmann [J/K]

# Escalas de Planck
ell_P = np.sqrt(hbar_SI * G_SI / c_SI**3)   # longitud de Planck [m]
t_P = np.sqrt(hbar_SI * G_SI / c_SI**5)     # tiempo de Planck [s]
m_P = np.sqrt(hbar_SI * c_SI / G_SI)        # masa de Planck [kg]
E_P = m_P * c_SI**2                          # energía de Planck [J]
rho_P = c_SI**5 / (hbar_SI * G_SI**2)       # densidad de Planck [kg/m³]

print(f'Escalas de Planck:')
print(f'  ℓ_P = {ell_P:.3e} m')
print(f'  t_P = {t_P:.3e} s')
print(f'  m_P = {m_P:.3e} kg')
print(f'  E_P = {E_P:.3e} J')
print(f'  ρ_P = {rho_P:.3e} kg/m³')
print()

# Valor observado de Λ (cosmológico)
# ρ_Λ = Λ c² / (8πG) pero usualmente se reporta como Ω_Λ · ρ_crit
# Planck 2018 + DESI: Ω_Λ ≈ 0.685, H_0 ≈ 67.4 km/s/Mpc
H0_SI = 67.4 * 1000 / (3.086e22)  # Hubble en s⁻¹
Omega_Lambda = 0.685
rho_crit = 3 * H0_SI**2 / (8 * np.pi * G_SI)   # kg/m³
rho_Lambda_obs = Omega_Lambda * rho_crit        # kg/m³

print(f'Valores cosmológicos observados:')
print(f'  H_0 = {H0_SI:.3e} s⁻¹')
print(f'  ρ_crit = {rho_crit:.3e} kg/m³')
print(f'  ρ_Λ_obs = {rho_Lambda_obs:.3e} kg/m³')
print()
print(f'Ratio clave:')
print(f'  ρ_Λ_obs / ρ_P = {rho_Lambda_obs / rho_P:.3e}')
print(f'  (este es el número pequeño famoso — del orden de 10⁻¹²³)')

## Paso 1 — Recalcular E_0 del cristal DEE

Reproducimos el cálculo de termodinámica pero con más autovalores para mejor estadística.

In [ ]:
# Funciones del cristal
def periodic_distance_matrix(points, L=1.0):
    diff = points[:, None, :] - points[None, :, :]
    diff = diff - L * np.round(diff / L)
    return np.linalg.norm(diff, axis=-1)

def grid_perturbed_T3(N_target, jitter_fraction, seed=42):
    rng = np.random.default_rng(seed)
    n_side = round(N_target**(1/3))
    N_actual = n_side**3
    spacing = 1.0 / n_side
    coords = np.arange(n_side) * spacing + spacing/2
    gx, gy, gz = np.meshgrid(coords, coords, coords, indexing='ij')
    points = np.stack([gx.ravel(), gy.ravel(), gz.ravel()], axis=1)
    points += rng.uniform(-jitter_fraction*spacing, jitter_fraction*spacing, size=points.shape)
    points = points % 1.0
    return points, N_actual

def build_laplacian(D_mat, k_neighbors=30, alpha=2.0):
    N = len(D_mat)
    D_search = D_mat.copy()
    np.fill_diagonal(D_search, np.inf)
    neighbor_idx = np.argpartition(D_search, k_neighbors, axis=1)[:, :k_neighbors]
    rows, cols, vals = [], [], []
    for i in range(N):
        for j in neighbor_idx[i]:
            d = D_mat[i, j]
            if d > 0:
                rows.append(i); cols.append(j); vals.append(1.0/d**alpha)
    W = csr_matrix((vals, (rows, cols)), shape=(N, N))
    W = (W + W.T) / 2
    degrees = np.array(W.sum(axis=1)).flatten()
    return diags(degrees) - W

# Construir cristal
N_TARGET = 1331
points, N = grid_perturbed_T3(N_TARGET, 0.1, seed=42)
D_mat = periodic_distance_matrix(points, L=1.0)
L_op = build_laplacian(D_mat, 30, alpha=2.0)

# Calcular muchos autovalores
N_EIGS = 600
print(f'Diagonalizando {N_EIGS} autovalores...')
t0 = time.time()
eigs_all, _ = eigsh(L_op, k=N_EIGS, which='SM', tol=1e-8)
eigs_all = np.sort(eigs_all)
eigs_nonzero = eigs_all[eigs_all > 1e-3]
omegas_nat = np.sqrt(eigs_nonzero)  # frecuencias en unidades naturales
print(f'Tiempo: {time.time()-t0:.1f}s')
print(f'Modos útiles: {len(omegas_nat)}')
print(f'Rango ω: [{omegas_nat.min():.3f}, {omegas_nat.max():.3f}]')

# Energía de punto cero total (en unidades naturales, ℏ=1)
E_0_total_natural = np.sum(0.5 * omegas_nat)
E_0_per_mode_natural = np.mean(0.5 * omegas_nat)

print(f'\nEnergía de punto cero (unidades naturales):')
print(f'  E_0 total = {E_0_total_natural:.2f}')
print(f'  E_0 por modo promedio = {E_0_per_mode_natural:.3f}')
print(f'  (coincide con el cálculo anterior de termodinámica ~18.8)')

## Paso 2 — Análisis paramétrico (Camino A)

Traducimos E_0 a unidades físicas. Dos parámetros del sustrato deben identificarse:

- **ℓ_c:** espaciado microscópico de la red cristalina [m]
- **c_s:** velocidad del sonido del cristal [m/s] — la identificamos con c

**Normalización de frecuencias:** en unidades naturales, ω_nat tiene valores ~10 a 50 (derivados del kernel 1/d²). La frecuencia física correspondiente es:

$$\omega_{phys} = \frac{c_s}{\ell_c} \cdot \omega_{nat}$$

Con densidad de modos ~1/ℓ_c³ por unidad de volumen y ℏω_phys por modo, la densidad de energía es:

$$\rho_\Lambda^{DEE} = \frac{1}{V_{\text{cell}}} \sum_n \frac{1}{2} \hbar \omega_{n,phys} = \frac{\hbar c_s}{2 \ell_c^4} \cdot \langle \omega_{nat} \rangle \cdot N_{modes}/V_{nat}$$

donde V_{nat} = 1 (cubo unidad en unidades naturales) y N_modes = cantidad de modos calculados.

In [ ]:
def rho_Lambda_DEE(ell_c, c_s=c_SI, omegas_natural=omegas_nat, V_natural=1.0):
    """
    Densidad de energía de vacío predicha por DEE.
    
    ell_c : espaciado de red [m]
    c_s   : velocidad del sonido del cristal [m/s]
    
    Retorna densidad de masa equivalente en kg/m³.
    """
    # Frecuencias físicas
    omega_scale = c_s / ell_c  # [rad/s]
    
    # Energía de punto cero total en unidades físicas
    E_0_total_phys = np.sum(0.5 * hbar_SI * omega_scale * omegas_natural)
    
    # Volumen físico correspondiente al cubo unidad del cálculo
    # En unidades naturales, V_nat = 1 y tiene N nodos
    # El volumen físico es (N^(1/3) · ell_c)³ = N · ell_c³
    N_nodes_total = len(omegas_natural) + 1  # +1 porque excluimos modo cero
    # Mejor: ajustar por ratio entre modos calculados y N total
    V_phys = N_nodes_total * ell_c**3
    
    rho_energy = E_0_total_phys / V_phys   # J/m³
    rho_mass = rho_energy / c_s**2         # kg/m³ equivalentes
    
    return rho_mass

# Barrido en ell_c
# Desde la longitud de Planck hasta 10^20 veces Planck
ell_c_range = np.logspace(-35, -15, 40)
rho_DEE_range = np.array([rho_Lambda_DEE(ell) for ell in ell_c_range])

# Plot
fig, ax = plt.subplots(figsize=(12, 7))
ax.loglog(ell_c_range, rho_DEE_range, 'b-', lw=2.5, label='ρ_Λ^DEE predicho')
ax.axhline(rho_Lambda_obs, color='red', linestyle='--', lw=2,
           label=f'ρ_Λ observado = {rho_Lambda_obs:.2e} kg/m³')
ax.axhline(rho_P, color='purple', linestyle=':', lw=1.5, alpha=0.7,
           label=f'ρ_Planck = {rho_P:.2e} kg/m³')
ax.axvline(ell_P, color='green', linestyle=':', lw=1.5, alpha=0.7,
           label=f'ℓ_Planck = {ell_P:.2e} m')

# Encontrar ell_c que reproduce ρ_Λ observada
idx_match = np.argmin(np.abs(rho_DEE_range - rho_Lambda_obs))
ell_c_fit = ell_c_range[idx_match]
ax.axvline(ell_c_fit, color='orange', linestyle='-.', lw=1.5, alpha=0.7,
           label=f'ℓ_c que ajusta = {ell_c_fit:.2e} m')

ax.set_xlabel('ℓ_c (espaciado microscópico) [m]', fontsize=13)
ax.set_ylabel('ρ_Λ predicho [kg/m³]', fontsize=13)
ax.set_title('Densidad de energía de vacío DEE vs ℓ_c (con c_s = c)', fontsize=13)
ax.legend(fontsize=11, loc='best')
ax.grid(alpha=0.3, which='both')
plt.tight_layout()
save_plot('34_rho_Lambda_vs_ell_c')
plt.show()

print(f'\nResultado del barrido paramétrico:')
print(f'  Para reproducir ρ_Λ observada ({rho_Lambda_obs:.2e} kg/m³),')
print(f'  se requiere ℓ_c ≈ {ell_c_fit:.3e} m')
print(f'  que es ℓ_c/ℓ_P = {ell_c_fit/ell_P:.3e} veces la longitud de Planck')

## Paso 3 — Evaluación directa con ℓ_c = ℓ_Planck (Camino B)

La elección más natural desde el punto de vista físico es ℓ_c = ℓ_Planck (escala fundamental de granularidad del espacio-tiempo). Esto es lo que asume Kleinert en el World Crystal Model. Calculamos qué predice DEE con esta elección y comparamos con Λ observada.

In [ ]:
rho_DEE_Planck = rho_Lambda_DEE(ell_P)
ratio_to_obs = rho_DEE_Planck / rho_Lambda_obs
ratio_to_Planck = rho_DEE_Planck / rho_P

print('='*70)
print('EVALUACIÓN DIRECTA con ℓ_c = ℓ_Planck, c_s = c')
print('='*70)
print()
print(f'  ρ_Λ^DEE (predicho)    = {rho_DEE_Planck:.3e} kg/m³')
print(f'  ρ_Λ observado         = {rho_Lambda_obs:.3e} kg/m³')
print(f'  ρ_Planck              = {rho_P:.3e} kg/m³')
print()
print(f'  Ratio DEE/observado    = {ratio_to_obs:.3e}')
print(f'  Ratio DEE/Planck       = {ratio_to_Planck:.3e}')
print()
print(f'  log₁₀(DEE/observado)   = {np.log10(ratio_to_obs):.1f}')
print(f'  log₁₀(DEE/Planck)      = {np.log10(ratio_to_Planck):.1f}')
print()
print('-'*70)
print('COMPARACIÓN CON EL PROBLEMA DE QFT ESTÁNDAR:')
print('-'*70)
print(f'  QFT estándar predice ρ_Λ ~ ρ_P')
print(f'  Discrepancia QFT: ρ_P/ρ_obs ≈ 10^{np.log10(rho_P/rho_Lambda_obs):.0f}')
print(f'  Discrepancia DEE: ρ_DEE/ρ_obs ≈ 10^{np.log10(rho_DEE_Planck/rho_Lambda_obs):.0f}')

## Paso 4 — Interpretación y mecanismos de ajuste

Si la discrepancia DEE-observado es grande (casi seguro), analizamos qué mecanismos podrían reducirla.

In [ ]:
# Gráfico: qué ℓ_c y qué c_s darían el resultado correcto
# ρ_Λ ∝ ℏ · c_s · ⟨ω_nat⟩ / ℓ_c⁴ / c_s² = ℏ · ⟨ω⟩ / (c_s · ℓ_c⁴)
# ρ_Λ ∝ 1/(c_s · ℓ_c⁴)

# Mapa 2D: variamos ambos parámetros y vemos cuándo ρ_Λ iguala la observada
ell_c_grid = np.logspace(-35, -10, 80)
c_s_grid = np.logspace(6, 10, 60)  # de 10^6 m/s a 10^10 m/s (c = 3·10^8)

ELL, CS = np.meshgrid(ell_c_grid, c_s_grid, indexing='ij')

# Función vectorizada
def rho_DEE_vec(ell, cs):
    omega_scale = cs / ell
    E_total = np.sum(0.5 * hbar_SI * omega_scale * omegas_nat)
    V = len(omegas_nat) * ell**3
    return E_total / V / cs**2

RHO = np.zeros_like(ELL)
for i in range(len(ell_c_grid)):
    for j in range(len(c_s_grid)):
        RHO[i, j] = rho_DEE_vec(ELL[i, j], CS[i, j])

log_RHO = np.log10(RHO)
log_target = np.log10(rho_Lambda_obs)

fig, ax = plt.subplots(figsize=(12, 8))
# Contornos de log(ρ_Λ)
levels = np.arange(-40, 120, 10)
cs_plot = ax.contour(ELL, CS, log_RHO, levels=levels, colors='gray', alpha=0.5)
ax.clabel(cs_plot, inline=True, fontsize=9, fmt='10^%d')

# Contorno específico de ρ_Λ observada (resaltado)
cs_target = ax.contour(ELL, CS, log_RHO, levels=[log_target],
                         colors='red', linewidths=3)
ax.clabel(cs_target, inline=True, fontsize=11, fmt='ρ_Λ obs.')

# Puntos de referencia
ax.plot(ell_P, c_SI, 'k*', markersize=20, label=f'(ℓ_P, c)')
ax.axvline(ell_P, color='green', linestyle=':', alpha=0.4)
ax.axhline(c_SI, color='blue', linestyle=':', alpha=0.4)

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('ℓ_c [m]', fontsize=13)
ax.set_ylabel('c_s [m/s]', fontsize=13)
ax.set_title('log₁₀(ρ_Λ^DEE) en el plano (ℓ_c, c_s)\nLínea roja: locus que reproduce ρ_Λ observada',
             fontsize=12)
ax.legend(fontsize=11)
ax.grid(alpha=0.3, which='both')
plt.tight_layout()
save_plot('35_mapa_parametrico')
plt.show()

## Paso 5 — El problema y posibles resoluciones

Si (y es altamente probable) la predicción con (ℓ_c, c_s) = (ℓ_P, c) sobrestima ρ_Λ por ~120 órdenes de magnitud, DEE está en el mismo problema que todas las otras teorías de vacío cuántico.

**Algunos mecanismos que podrían restaurar el acuerdo:**

1. **Cutoff efectivo menor que Planck:** si solo los modos por debajo de cierta ω_max contribuyen al vacío efectivo, ρ_Λ se reduce.

2. **Cancelación entre sectores:** si hay contribuciones negativas (de solitones, por ejemplo) que cancelan la mayor parte de E_0.

3. **Renormalización del sustrato:** la E_0 "cruda" del cristal se sustrae como parte de la definición del vacío observable.

4. **Parámetros distintos de Planck:** si ℓ_c no es ℓ_P sino una escala intermedia.

Calculamos qué cutoff ω_max sería necesario para reducir ρ_Λ^DEE al valor observado.

In [ ]:
# Mecanismo 1: cutoff efectivo
# Si solo integramos modos por debajo de ω_cutoff, ¿qué ω_cutoff da ρ_Λ observada?

def rho_with_cutoff(omegas_natural, omega_cutoff_natural, ell_c=ell_P, c_s=c_SI):
    omega_scale = c_s / ell_c
    mask = omegas_natural < omega_cutoff_natural
    omegas_used = omegas_natural[mask]
    if len(omegas_used) == 0:
        return 0.0
    E_total = np.sum(0.5 * hbar_SI * omega_scale * omegas_used)
    V = len(omegas_natural) * ell_c**3  # mantenemos V constante (es el volumen del sistema)
    return E_total / V / c_s**2

# Barrido de cutoff
cutoffs_natural = np.logspace(-5, 2, 60)
rho_cutoff = np.array([rho_with_cutoff(omegas_nat, oc) for oc in cutoffs_natural])

# ω_cutoff físico correspondiente
cutoffs_phys = cutoffs_natural * c_SI / ell_P

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.loglog(cutoffs_natural, rho_cutoff, 'b-', lw=2, label='ρ_Λ(ω_cutoff)')
ax.axhline(rho_Lambda_obs, color='red', linestyle='--', lw=2, label='ρ_Λ observada')
ax.set_xlabel('ω_cutoff (unidades naturales del cristal)', fontsize=12)
ax.set_ylabel('ρ_Λ [kg/m³]', fontsize=12)
ax.set_title('Con cutoff: densidad de energía como función de ω_max', fontsize=12)
ax.legend(fontsize=10)
ax.grid(alpha=0.3, which='both')

ax = axes[1]
ax.loglog(cutoffs_phys, rho_cutoff, 'g-', lw=2)
ax.axhline(rho_Lambda_obs, color='red', linestyle='--', lw=2, label='ρ_Λ observada')
# Agregar marcadores de escalas conocidas
omega_P = c_SI / ell_P
ax.axvline(omega_P, color='purple', linestyle=':', label=f'ω_Planck')
ax.axvline(H0_SI, color='orange', linestyle=':', label=f'H_0')
ax.set_xlabel('ω_cutoff [rad/s]', fontsize=12)
ax.set_ylabel('ρ_Λ [kg/m³]', fontsize=12)
ax.set_title('Mismo plot en escala física', fontsize=12)
ax.legend(fontsize=10)
ax.grid(alpha=0.3, which='both')

plt.tight_layout()
save_plot('36_cutoff_efectivo')
plt.show()

# Encontrar cutoff que da ρ_Λ observada
idx_match_cut = np.argmin(np.abs(rho_cutoff - rho_Lambda_obs))
omega_cut_fit = cutoffs_natural[idx_match_cut]
omega_cut_phys = cutoffs_phys[idx_match_cut]

print(f'Cutoff efectivo que reproduce ρ_Λ observada:')
print(f'  ω_cutoff (natural) = {omega_cut_fit:.3e}')
print(f'  ω_cutoff (física)  = {omega_cut_phys:.3e} rad/s')
print(f'  Ratio a ω_Planck = {omega_cut_phys/omega_P:.3e}')
print(f'  Ratio a H_0      = {omega_cut_phys/H0_SI:.3e}')

## Síntesis

In [ ]:
print('='*75)
print('SÍNTESIS — Vacío cuántico de DEE y constante cosmológica')
print('='*75)
print()
print('LO QUE SÍ DEMOSTRAMOS:')
print(f'  1. El cristal DEE tiene energía de punto cero no nula: E_0 ~ {E_0_total_natural:.1f}')
print(f'     en unidades naturales (~{E_0_per_mode_natural:.2f} por modo)')
print()
print(f'  2. Traduciendo a unidades físicas con ℓ_c = ℓ_Planck:')
print(f'     ρ_Λ^DEE = {rho_DEE_Planck:.3e} kg/m³')
print()
print(f'  3. La discrepancia con observación es:')
print(f'     log₁₀(DEE/obs) = {np.log10(ratio_to_obs):.1f}')
print(f'     (comparable al problema tradicional de QFT: ~120 órdenes)')
print()
print('LO QUE NO DEMOSTRAMOS:')
print(f'  1. DEE no "resuelve" el problema de Λ; está en la misma dificultad que QFT')
print()
print(f'  2. Para reproducir ρ_Λ observada, DEE requeriría:')
print(f'     - ℓ_c ≈ {ell_c_fit:.2e} m (si c_s = c) — NO ES PLANCK')
print(f'     - o un cutoff efectivo en ω ≈ {omega_cut_phys:.2e} rad/s')
print(f'     - o algún mecanismo de cancelación no testeado')
print()
print('CONCLUSIÓN HONESTA:')
print('  DEE tiene la ventaja sobre QFT de identificar la fuente del vacío con')
print('  un objeto concreto (el cristal cuantizado). Pero el problema numérico')
print('  persiste. Para que DEE reproduzca Λ_obs se requiere un mecanismo')
print('  adicional (cutoff, cancelación topológica, renormalización) que no está')
print('  contenido en el modelo actual.')
print()
print('  Este resultado es una PREDICCIÓN CONCRETA del modelo, no un ajuste.')
print('  Que la predicción naive no coincida con Λ_obs es UN PROBLEMA ABIERTO.')

In [ ]:
import shutil
shutil.make_archive('plots_vacio_Lambda', 'zip', PLOT_DIR)
print(f'\nZip creado: plots_vacio_Lambda.zip')

try:
    from google.colab import files
    files.download('plots_vacio_Lambda.zip')
    print('→ Descarga iniciada')
except ImportError:
    pass